# NYCU Data Mining Assignment 3 (Kaggle Notebook v7.4 Final Submit)

This notebook keeps the v7 tabular and CNN+BiGRU sequence branches, then learns a meta-combination from out-of-fold probabilities using a LightGBM meta model. It writes `submission_v74_final_submit.csv`.


In [ ]:
import gc
import math
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import iqr, kurtosis, skew
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 3
CHUNK_COUNT = 6
BASE_COLS = ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']
AXIS_COLS = ['mean_x', 'mean_y', 'mean_z']
STD_COLS = ['std_x', 'std_y', 'std_z']

TABULAR_LGBM_WEIGHT = 0.7
TABULAR_XGB_WEIGHT = 0.3
HYBRID_TABULAR_WEIGHT = 0.75
HYBRID_SEQUENCE_WEIGHT = 0.25

BATCH_SIZE = 128
EPOCHS = 14
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4
NUM_WORKERS = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQUENCE_COLS = ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']
USE_EXTRA_SEQUENCE_CHANNELS = True

def seed_everything(seed: int = RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()
print('DEVICE:', DEVICE)


In [ ]:
def discover_data_root() -> Path:
    search_roots = [Path('/kaggle/input'), Path.cwd()]
    for root in search_roots:
        if not root.exists():
            continue
        for sample_path in root.rglob('sample_submission.csv'):
            candidate = sample_path.parent
            if (candidate / 'train' / 'train').exists() and (candidate / 'test' / 'test').exists():
                return candidate
    raise FileNotFoundError('Could not find sample_submission.csv with train/train and test/test folders.')

DATA_ROOT = discover_data_root()
TRAIN_DIR = DATA_ROOT / 'train' / 'train'
TEST_DIR = DATA_ROOT / 'test' / 'test'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
TABULAR_CACHE_DIR = WORK_DIR / 'feature_cache_v71_tabular'
SEQUENCE_CACHE_DIR = WORK_DIR / 'sequence_cache_v71'
TABULAR_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_paths = sorted(TRAIN_DIR.glob('User_*/*.csv'))
test_paths = sorted(TEST_DIR.glob('User_*/*.csv'))
sample_submission = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'Train files: {len(train_paths):,}')
print(f'Test files: {len(test_paths):,}')


In [ ]:
def safe_stat(value: float) -> float:
    if pd.isna(value) or np.isinf(value):
        return 0.0
    return float(value)

def lag_autocorr(values: np.ndarray, lag: int) -> float:
    if len(values) <= lag:
        return 0.0
    x0 = values[:-lag] - values[:-lag].mean()
    x1 = values[lag:] - values[lag:].mean()
    denom = np.sqrt((x0 ** 2).sum() * (x1 ** 2).sum())
    if denom == 0:
        return 0.0
    return safe_stat((x0 * x1).sum() / denom)

def top_fft(values: np.ndarray, k: int = 3) -> np.ndarray:
    fft_values = np.abs(np.fft.rfft(values.astype(np.float32)))
    if len(fft_values) > 0:
        fft_values[0] = 0.0
    top = np.sort(fft_values)[-k:][::-1]
    if len(top) < k:
        top = np.pad(top, (0, k - len(top)), constant_values=0.0)
    return top

def summarize(values: np.ndarray, prefix: str) -> dict:
    values = np.asarray(values, dtype=np.float32)
    diffs = np.diff(values) if len(values) > 1 else np.array([0.0], dtype=np.float32)
    q10, q25, q50, q75, q90 = np.quantile(values, [0.1, 0.25, 0.5, 0.75, 0.9])
    fft_top = top_fft(values, 3)
    return {
        f'{prefix}_mean': safe_stat(values.mean()),
        f'{prefix}_std': safe_stat(values.std()),
        f'{prefix}_min': safe_stat(values.min()),
        f'{prefix}_max': safe_stat(values.max()),
        f'{prefix}_q10': safe_stat(q10),
        f'{prefix}_q25': safe_stat(q25),
        f'{prefix}_median': safe_stat(q50),
        f'{prefix}_q75': safe_stat(q75),
        f'{prefix}_q90': safe_stat(q90),
        f'{prefix}_iqr': safe_stat(iqr(values)),
        f'{prefix}_range': safe_stat(values.max() - values.min()),
        f'{prefix}_energy': safe_stat(np.mean(values ** 2)),
        f'{prefix}_diff_mean': safe_stat(diffs.mean()),
        f'{prefix}_diff_std': safe_stat(diffs.std()),
        f'{prefix}_diff_abs_mean': safe_stat(np.abs(diffs).mean()),
        f'{prefix}_autocorr1': lag_autocorr(values, 1),
        f'{prefix}_autocorr5': lag_autocorr(values, 5),
        f'{prefix}_skew': safe_stat(skew(values, bias=False)) if values.std() > 1e-8 else 0.0,
        f'{prefix}_kurtosis': safe_stat(kurtosis(values, bias=False)) if values.std() > 1e-8 else 0.0,
        f'{prefix}_fft_top1': safe_stat(fft_top[0]),
        f'{prefix}_fft_top2': safe_stat(fft_top[1]),
        f'{prefix}_fft_top3': safe_stat(fft_top[2]),
    }

def spectral_band_features(values: np.ndarray, prefix: str) -> dict:
    values = np.asarray(values, dtype=np.float32)
    power = np.abs(np.fft.rfft(values)) ** 2
    if len(power) == 0 or power.sum() == 0:
        return {
            f'{prefix}_spec_entropy': 0.0,
            f'{prefix}_spec_dom_ratio': 0.0,
            f'{prefix}_spec_low_ratio': 0.0,
            f'{prefix}_spec_mid_ratio': 0.0,
            f'{prefix}_spec_high_ratio': 0.0,
        }
    power[0] = 0.0
    total = power.sum()
    n = len(power)
    low_end = max(1, n // 6)
    mid_end = max(low_end + 1, n // 3)
    low = power[:low_end].sum()
    mid = power[low_end:mid_end].sum()
    high = power[mid_end:].sum()
    prob = power / total
    prob = prob[prob > 0]
    entropy = -(prob * np.log(prob)).sum()
    return {
        f'{prefix}_spec_entropy': safe_stat(entropy),
        f'{prefix}_spec_dom_ratio': safe_stat(power.max() / total),
        f'{prefix}_spec_low_ratio': safe_stat(low / total),
        f'{prefix}_spec_mid_ratio': safe_stat(mid / total),
        f'{prefix}_spec_high_ratio': safe_stat(high / total),
    }

def motion_burst_features(values: np.ndarray, prefix: str) -> dict:
    values = np.asarray(values, dtype=np.float32)
    threshold = values.mean() + values.std()
    active = values > threshold
    peak_count = 0
    for i in range(1, len(values) - 1):
        if values[i] > values[i - 1] and values[i] > values[i + 1] and values[i] > threshold:
            peak_count += 1
    longest_run = 0
    current = 0
    for flag in active:
        if flag:
            current += 1
            longest_run = max(longest_run, current)
        else:
            current = 0
    return {
        f'{prefix}_active_ratio': safe_stat(active.mean()),
        f'{prefix}_active_run': float(longest_run),
        f'{prefix}_peak_count': float(peak_count),
    }

def normalize_sequence(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    std = values.std()
    if std < 1e-8:
        return np.zeros_like(values)
    return (values - values.mean()) / std

def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values('index').reset_index(drop=True).copy()
    out['acc_mag_mean'] = np.sqrt((out[AXIS_COLS] ** 2).sum(axis=1))
    out['acc_mag_std'] = np.sqrt((out[STD_COLS] ** 2).sum(axis=1))
    out['jerk_x'] = out['mean_x'].diff().fillna(0.0)
    out['jerk_y'] = out['mean_y'].diff().fillna(0.0)
    out['jerk_z'] = out['mean_z'].diff().fillna(0.0)
    out['jerk_mag'] = np.sqrt(out['jerk_x'] ** 2 + out['jerk_y'] ** 2 + out['jerk_z'] ** 2)
    return out

def extract_features(df: pd.DataFrame) -> dict:
    df = add_derived(df)
    features = {'row_count': int(len(df)), 'index_max': int(df['index'].max())}
    feature_cols = BASE_COLS + ['acc_mag_mean', 'acc_mag_std', 'jerk_x', 'jerk_y', 'jerk_z', 'jerk_mag']
    for col in feature_cols:
        features.update(summarize(df[col].to_numpy(), col))
    for col in ['acc_mag_mean', 'jerk_mag']:
        values = df[col].to_numpy(dtype=np.float32)
        features.update(spectral_band_features(values, col))
        features.update(motion_burst_features(values, col))
    for col in ['mean_x', 'mean_y', 'mean_z', 'acc_mag_mean', 'jerk_mag']:
        norm_values = normalize_sequence(df[col].to_numpy())
        features.update(summarize(norm_values, f'{col}_norm'))
    for left, right in [('mean_x', 'mean_y'), ('mean_y', 'mean_z'), ('mean_z', 'mean_x'), ('std_x', 'std_y'), ('std_y', 'std_z'), ('std_z', 'std_x')]:
        features[f'corr_{left}_{right}'] = safe_stat(df[left].corr(df[right]))
    chunk_size = math.ceil(len(df) / CHUNK_COUNT)
    chunk_cols = ['acc_mag_mean', 'acc_mag_std', 'jerk_mag', 'mean_x', 'mean_y', 'mean_z']
    for chunk_idx in range(CHUNK_COUNT):
        chunk = df.iloc[chunk_idx * chunk_size:(chunk_idx + 1) * chunk_size]
        if chunk.empty:
            chunk = df.iloc[-1:]
        for col in chunk_cols:
            features[f'{col}_chunk{chunk_idx}_mean'] = safe_stat(chunk[col].mean())
            features[f'{col}_chunk{chunk_idx}_std'] = safe_stat(chunk[col].std())
    first_half = df.iloc[: len(df) // 2]
    second_half = df.iloc[len(df) // 2 :]
    for col in ['acc_mag_mean', 'acc_mag_std', 'jerk_mag', 'mean_x', 'mean_y', 'mean_z']:
        features[f'{col}_half_gap'] = safe_stat(second_half[col].mean() - first_half[col].mean())
    return features

def build_feature_table(paths, cache_path: Path, is_train: bool) -> pd.DataFrame:
    if cache_path.exists():
        print(f'Loading tabular cache from {cache_path}')
        return pd.read_pickle(cache_path)
    rows = []
    iterator = tqdm(paths, total=len(paths), desc='Extracting train features' if is_train else 'Extracting test features')
    for path in iterator:
        df = pd.read_csv(path)
        row = extract_features(df)
        row['file_id'] = int(df['file_id'].iloc[0])
        row['user_name'] = path.parent.name
        if is_train:
            row['label'] = int(df['label'].iloc[0])
        rows.append(row)
    feature_df = pd.DataFrame(rows).sort_values('file_id').reset_index(drop=True)
    feature_df.to_pickle(cache_path)
    return feature_df

train_df = build_feature_table(train_paths, TABULAR_CACHE_DIR / 'train_features.pkl', is_train=True)
test_df = build_feature_table(test_paths, TABULAR_CACHE_DIR / 'test_features.pkl', is_train=False)


In [ ]:
def build_sequence(df: pd.DataFrame) -> np.ndarray:
    df = df.sort_values('index').reset_index(drop=True).copy()
    channels = [df[col].to_numpy(dtype=np.float32) for col in SEQUENCE_COLS]
    if USE_EXTRA_SEQUENCE_CHANNELS:
        acc_mag_mean = np.sqrt((df[['mean_x', 'mean_y', 'mean_z']] ** 2).sum(axis=1)).to_numpy(dtype=np.float32)
        jerk_xyz = df[['mean_x', 'mean_y', 'mean_z']].diff().fillna(0.0)
        jerk_mag = np.sqrt((jerk_xyz ** 2).sum(axis=1)).to_numpy(dtype=np.float32)
        channels.extend([acc_mag_mean, jerk_mag])
    seq = np.stack(channels, axis=0)
    mean = seq.mean(axis=1, keepdims=True)
    std = seq.std(axis=1, keepdims=True)
    seq = (seq - mean) / np.clip(std, 1e-6, None)
    return seq.astype(np.float32)

def build_sequence_table(paths, cache_path: Path, is_train: bool):
    if cache_path.exists():
        loaded = np.load(cache_path, allow_pickle=True)
        return loaded['X'], loaded['file_ids'], loaded['users'], loaded['y'] if is_train else None
    X_list, file_ids, users, y_list = [], [], [], []
    iterator = tqdm(paths, total=len(paths), desc='Building train sequences' if is_train else 'Building test sequences')
    for path in iterator:
        df = pd.read_csv(path)
        X_list.append(build_sequence(df))
        file_ids.append(int(df['file_id'].iloc[0]))
        users.append(path.parent.name)
        if is_train:
            y_list.append(int(df['label'].iloc[0]))
    X = np.stack(X_list).astype(np.float32)
    file_ids = np.asarray(file_ids)
    users = np.asarray(users)
    y = np.asarray(y_list, dtype=np.int64) if is_train else None
    np.savez_compressed(cache_path, X=X, file_ids=file_ids, users=users, y=y)
    return X, file_ids, users, y

X_train_seq, train_file_ids, train_users, y_seq = build_sequence_table(train_paths, SEQUENCE_CACHE_DIR / 'train_sequences.npz', is_train=True)
X_test_seq, test_file_ids, test_users, _ = build_sequence_table(test_paths, SEQUENCE_CACHE_DIR / 'test_sequences.npz', is_train=False)


In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

class CNNBiGRUClassifier(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.15),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )
        self.bigru = nn.GRU(128, 128, batch_first=True, bidirectional=True)
        self.attn = nn.Sequential(nn.Linear(256, 128), nn.Tanh(), nn.Linear(128, 1))
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        x = self.cnn(x)
        x = x.transpose(1, 2)
        gru_out, _ = self.bigru(x)
        attn_score = torch.softmax(self.attn(gru_out), dim=1)
        attn_feat = (gru_out * attn_score).sum(dim=1)
        avg_feat = gru_out.mean(dim=1)
        feat = torch.cat([attn_feat, avg_feat], dim=1)
        return self.classifier(feat)

def make_loaders(X_tr, y_tr, X_va, y_va):
    train_ds = SequenceDataset(X_tr, y_tr)
    valid_ds = SequenceDataset(X_va, y_va)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, valid_loader

def predict_loader(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, (tuple, list)):
                batch = batch[0]
            batch = batch.to(DEVICE)
            logits = model(batch)
            probs.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
    return np.concatenate(probs, axis=0)

def train_sequence_fold(X_tr, y_tr, X_va, y_va, num_classes):
    train_loader, valid_loader = make_loaders(X_tr, y_tr, X_va, y_va)
    model = CNNBiGRUClassifier(in_channels=X_tr.shape[1], num_classes=num_classes).to(DEVICE)
    class_counts = np.bincount(y_tr, minlength=num_classes)
    class_weights = class_counts.sum() / np.maximum(class_counts, 1)
    class_weights = class_weights / class_weights.mean()
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best_score = -1.0
    best_state = None
    patience_left = PATIENCE
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
        valid_probs = predict_loader(model, valid_loader)
        valid_pred = valid_probs.argmax(axis=1)
        valid_score = f1_score(y_va, valid_pred, average='macro')
        scheduler.step(valid_score)
        print(f'    seq epoch {epoch:02d} | train_loss={np.mean(losses):.4f} | valid_macro_f1={valid_score:.5f}')
        if valid_score > best_score:
            best_score = valid_score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                print('    seq early stopping triggered')
                break
    model.load_state_dict(best_state)
    return model


In [ ]:
feature_cols = [c for c in train_df.columns if c not in ['file_id', 'user_name', 'label']]
X_tab = train_df[feature_cols]
X_tab_test = test_df[feature_cols]
y = train_df['label'].astype(int).to_numpy()
groups = train_df['user_name'].to_numpy()
num_classes = len(np.unique(y))

assert np.array_equal(train_df['file_id'].to_numpy(), train_file_ids)
assert np.array_equal(test_df['file_id'].to_numpy(), test_file_ids)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

def build_tabular_models():
    return [
        ('lgbm', TABULAR_LGBM_WEIGHT, LGBMClassifier(
            n_estimators=480,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=8,
            min_child_samples=20,
            subsample=0.85,
            colsample_bytree=0.6,
            reg_alpha=0.2,
            reg_lambda=1.0,
            objective='multiclass',
            random_state=RANDOM_STATE,
            class_weight='balanced',
            n_jobs=-1,
            force_col_wise=True,
            verbosity=-1,
        )),
        ('xgb', TABULAR_XGB_WEIGHT, XGBClassifier(
            n_estimators=480,
            learning_rate=0.04,
            max_depth=6,
            min_child_weight=2,
            subsample=0.8,
            colsample_bytree=0.65,
            reg_alpha=0.05,
            reg_lambda=1.0,
            objective='multi:softprob',
            eval_metric='mlogloss',
            tree_method='hist',
            random_state=RANDOM_STATE,
            num_class=num_classes,
            n_jobs=-1,
        )),
    ]

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_tab_probs = np.zeros((len(train_df), num_classes), dtype=np.float32)
oof_seq_probs = np.zeros((len(train_df), num_classes), dtype=np.float32)
oof_hybrid_probs = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_tab_probs = np.zeros((len(test_df), num_classes), dtype=np.float32)
test_seq_probs = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_tab, y, groups), start=1):
    print(f'\n===== Fold {fold} =====')
    X_train_tab, X_valid_tab = X_tab.iloc[train_idx], X_tab.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]
    fold_tab_valid = np.zeros((len(valid_idx), num_classes), dtype=np.float32)
    fold_tab_test = np.zeros((len(test_df), num_classes), dtype=np.float32)
    for model_name, weight, model in build_tabular_models():
        pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', model)])
        pipeline.fit(X_train_tab, y_train)
        valid_probs = pipeline.predict_proba(X_valid_tab).astype(np.float32)
        valid_pred = valid_probs.argmax(axis=1)
        score = f1_score(y_valid, valid_pred, average='macro')
        print(f'  {model_name}: macro F1 = {score:.5f}, weight = {weight:.2f}')
        fold_tab_valid += weight * valid_probs
        fold_tab_test += weight * pipeline.predict_proba(X_tab_test).astype(np.float32)
        del pipeline
        gc.collect()
    tab_score = f1_score(y_valid, fold_tab_valid.argmax(axis=1), average='macro')
    print(f'  tabular ensemble macro F1 = {tab_score:.5f}')

    seq_model = train_sequence_fold(X_train_seq[train_idx], y_train, X_train_seq[valid_idx], y_valid, num_classes)
    valid_seq_loader = DataLoader(SequenceDataset(X_train_seq[valid_idx], y_valid), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_seq_loader = DataLoader(SequenceDataset(X_test_seq), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    fold_seq_valid = predict_loader(seq_model, valid_seq_loader)
    fold_seq_test = predict_loader(seq_model, test_seq_loader)
    seq_score = f1_score(y_valid, fold_seq_valid.argmax(axis=1), average='macro')
    print(f'  sequence macro F1 = {seq_score:.5f}')

    fold_hybrid_valid = HYBRID_TABULAR_WEIGHT * fold_tab_valid + HYBRID_SEQUENCE_WEIGHT * fold_seq_valid
    hybrid_score = f1_score(y_valid, fold_hybrid_valid.argmax(axis=1), average='macro')
    print(f'  manual hybrid macro F1 = {hybrid_score:.5f}')

    oof_tab_probs[valid_idx] = fold_tab_valid
    oof_seq_probs[valid_idx] = fold_seq_valid
    oof_hybrid_probs[valid_idx] = fold_hybrid_valid
    test_tab_probs += fold_tab_test / N_SPLITS
    test_seq_probs += fold_seq_test / N_SPLITS

    del seq_model, valid_seq_loader, test_seq_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tab_oof_score = f1_score(y, oof_tab_probs.argmax(axis=1), average='macro')
seq_oof_score = f1_score(y, oof_seq_probs.argmax(axis=1), average='macro')
hybrid_oof_score = f1_score(y, oof_hybrid_probs.argmax(axis=1), average='macro')
print(f'\nTabular OOF macro F1: {tab_oof_score:.5f}')
print(f'Sequence OOF macro F1: {seq_oof_score:.5f}')
print(f'Manual Hybrid OOF macro F1: {hybrid_oof_score:.5f}')

def build_stack_features(tab_probs, seq_probs):
    tab_top1 = tab_probs.max(axis=1, keepdims=True)
    seq_top1 = seq_probs.max(axis=1, keepdims=True)

    tab_sorted = np.sort(tab_probs, axis=1)
    seq_sorted = np.sort(seq_probs, axis=1)

    tab_top2 = tab_sorted[:, -2:-1]
    seq_top2 = seq_sorted[:, -2:-1]

    tab_margin = tab_top1 - tab_top2
    seq_margin = seq_top1 - seq_top2

    tab_entropy = -(tab_probs * np.log(tab_probs + 1e-8)).sum(axis=1, keepdims=True)
    seq_entropy = -(seq_probs * np.log(seq_probs + 1e-8)).sum(axis=1, keepdims=True)

    agree = (
        np.argmax(tab_probs, axis=1) == np.argmax(seq_probs, axis=1)
    ).astype(np.float32).reshape(-1, 1)

    return np.hstack([
        tab_probs,
        seq_probs,
        tab_top1,
        seq_top1,
        tab_top2,
        seq_top2,
        tab_margin,
        seq_margin,
        tab_entropy,
        seq_entropy,
        agree,
    ]).astype(np.float32)

stack_train = build_stack_features(oof_tab_probs, oof_seq_probs)
stack_test = build_stack_features(test_tab_probs, test_seq_probs)

meta_model = LGBMClassifier(
    objective='multiclass',
    num_class=num_classes,
    n_estimators=180,
    learning_rate=0.04,
    num_leaves=15,
    max_depth=4,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.2,
    reg_lambda=1.5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
    force_col_wise=True,
)
meta_model.fit(stack_train, y)
stack_oof_probs = meta_model.predict_proba(stack_train).astype(np.float32)
stack_test_probs = meta_model.predict_proba(stack_test).astype(np.float32)
stack_oof_score = f1_score(y, stack_oof_probs.argmax(axis=1), average='macro')
print(f'Meta-stacking OOF macro F1: {stack_oof_score:.5f}')


In [ ]:
submission = sample_submission.copy()
submission['Label'] = stack_test_probs.argmax(axis=1)
submission.to_csv(WORK_DIR / 'submission_v74_final_submit.csv', index=False)
submission.head()
